In [ ]:
import scglue
import scanpy as sc
import infercnvpy as cnv
from torchgen.api.types import layoutT

from helper.annotation_helper import *
import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import helper.plot_helper as hp
hp.setMyStyle()

In [ ]:
os.getcwd()

In [ ]:
myStyle()
os.chdir('/mnt/d/lxk/project/jiangshucai20260506')

In [ ]:
rna = sc.read_h5ad('./h5ad/rna_0.h5ad')

In [ ]:
sc.pl.umap(rna, color="celltype")

In [ ]:
scglue.data.get_gene_annotation(
    rna, gtf="./genes.gtf.gz",
    gtf_by="gene_name"
)
rna.var.loc[:, ["chrom", "chromStart", "chromEnd"]].head()
rna = rna[:, ~rna.var['chromStart'].isna()].copy()

In [ ]:
rna.var['chromosome'] = rna.var['chrom']
rna.var['start'] = rna.var['chromStart']
rna.var['end'] = rna.var['chromEnd']
rna.var.loc[:, ["chromosome", "start", "end"]].head()

In [ ]:
np.unique(rna.obs['celltype'])

In [ ]:
cnv.tl.infercnv(
    rna,
    layer = 'normalize',
    #step = 1,
    reference_key="celltype",
    reference_cat = [
    "Endo",
    "Mast",
    "Myeloid",
    "Pericyte/SMC",
    "T/NK",
],
    window_size=250,
)

In [ ]:
rna.obs_keys

In [ ]:
cnv.pl.chromosome_heatmap(rna, groupby="sample",save='heatmap.pdf')


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import ListedColormap
from scipy import sparse


def plot_infercnv_heatmap(
    adata,
    cnv_key="X_cnv",
    groupby="celltype",
    outdir="./fig/infercnv",
    filename="infercnv_custom",
    max_cells_per_group=800,
    vmin=-0.25,
    vmax=0.25,
    figsize=(17, 9),
    random_state=2026,
):
    os.makedirs(outdir, exist_ok=True)
    rng = np.random.default_rng(random_state)

    # CNV matrix: cells x CNV windows
    cnv = adata.obsm[cnv_key]
    if sparse.issparse(cnv):
        cnv = cnv.toarray()
    cnv = np.asarray(cnv)

    obs = adata.obs.copy()
    obs[groupby] = obs[groupby].astype(str)

    # order groups by existing categorical order if available
    if hasattr(adata.obs[groupby], "cat"):
        group_order = adata.obs[groupby].cat.categories.astype(str).tolist()
        group_order = [g for g in group_order if g in set(obs[groupby])]
    else:
        group_order = obs[groupby].value_counts().index.tolist()

    # downsample cells per group
    selected_cells = []

    for g in group_order:
        cells = obs.index[obs[groupby] == g].to_numpy()
        if len(cells) == 0:
            continue
        if len(cells) > max_cells_per_group:
            cells = rng.choice(cells, max_cells_per_group, replace=False)
        selected_cells.extend(cells.tolist())

    cell_idx = adata.obs_names.get_indexer(selected_cells)
    valid = cell_idx >= 0
    cell_idx = cell_idx[valid]
    selected_cells = np.array(selected_cells)[valid].tolist()

    mat = cnv[cell_idx, :]
    mat = np.clip(mat, vmin, vmax)

    obs_sel = obs.loc[selected_cells].copy()

    # group colors
    if f"{groupby}_colors" in adata.uns and hasattr(adata.obs[groupby], "cat"):
        cats = adata.obs[groupby].cat.categories.astype(str).tolist()
        colors = adata.uns[f"{groupby}_colors"]
        group_color_map = dict(zip(cats, colors))
        group_color_map = {
            k: v for k, v in group_color_map.items()
            if k in set(obs_sel[groupby])
        }
    else:
        palette = plt.cm.tab20(np.linspace(0, 1, len(group_order)))
        group_color_map = dict(zip(group_order, palette))

    group_order_present = [g for g in group_order if g in set(obs_sel[groupby])]
    group_color_map = {
        g: group_color_map.get(g, "#999999")
        for g in group_order_present
    }

    group_codes = pd.Categorical(
        obs_sel[groupby],
        categories=group_order_present,
        ordered=True,
    ).codes

    group_cmap = ListedColormap(
        [group_color_map[g] for g in group_order_present]
    )

    # approximate chromosome bar from cnv windows
    # because X_cnv is window-level, exact gene-level chromosome labels are not available here
    n_windows = mat.shape[1]
    chrom_order = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
    n_chr = len(chrom_order)

    # estimate chromosome boundaries from gene annotation, then scale to CNV windows
    var = adata.var.copy()
    chr_col = "chromosome" if "chromosome" in var.columns else "chrom"
    start_col = "start" if "start" in var.columns else "chromStart"

    var = var[var[chr_col].notna() & var[start_col].notna()].copy()
    var[chr_col] = var[chr_col].astype(str)
    var = var[var[chr_col].isin(chrom_order)].copy()
    var[chr_col] = pd.Categorical(var[chr_col], categories=chrom_order, ordered=True)
    var = var.sort_values([chr_col, start_col])

    gene_counts = var[chr_col].value_counts(sort=False).reindex(chrom_order).fillna(0).to_numpy()
    if gene_counts.sum() > 0:
        proportions = gene_counts / gene_counts.sum()
        widths = np.maximum(np.round(proportions * n_windows).astype(int), 1)
        diff = n_windows - widths.sum()
        widths[np.argmax(widths)] += diff
        widths = np.maximum(widths, 1)
        # fix possible rounding overflow
        while widths.sum() > n_windows:
            i = np.argmax(widths)
            widths[i] -= 1
        while widths.sum() < n_windows:
            i = np.argmax(widths)
            widths[i] += 1
    else:
        widths = np.repeat(n_windows // n_chr, n_chr)
        widths[-1] += n_windows - widths.sum()

    chrom_codes = np.concatenate([
        np.repeat(i, w) for i, w in enumerate(widths)
    ])[:n_windows]

    boundaries = np.cumsum(widths)
    centers = boundaries - widths / 2
    labels = [c.replace("chr", "") for c in chrom_order]

    chrom_palette = [
        "#D9E3F0", "#F3E1D3"
    ] * 12
    chrom_cmap = ListedColormap(chrom_palette[:n_chr])

    # plot
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(
        nrows=3,
        ncols=3,
        width_ratios=[0.28, 14, 0.45],
        height_ratios=[0.32, 0.08, 8],
        wspace=0.03,
        hspace=0.02,
    )

    ax_chr = fig.add_subplot(gs[0, 1])
    ax_group = fig.add_subplot(gs[2, 0])
    ax_heat = fig.add_subplot(gs[2, 1])
    ax_cbar = fig.add_subplot(gs[2, 2])

    # chromosome annotation bar
    ax_chr.imshow(
        chrom_codes[np.newaxis, :],
        aspect="auto",
        interpolation="nearest",
        cmap=chrom_cmap,
        vmin=0,
        vmax=n_chr - 1,
    )
    ax_chr.set_xticks(centers)
    ax_chr.set_xticklabels(labels, fontsize=7)
    ax_chr.set_yticks([])
    ax_chr.tick_params(axis="x", length=0)
    ax_chr.set_title("Chromosome", fontsize=10, pad=4)

    for b in boundaries:
        ax_chr.axvline(b, color="white", linewidth=0.5)

    # group side bar
    ax_group.imshow(
        group_codes[:, np.newaxis],
        aspect="auto",
        interpolation="nearest",
        cmap=group_cmap,
        vmin=0,
        vmax=max(group_codes) if len(group_codes) else 1,
    )
    ax_group.set_xticks([])
    ax_group.set_yticks([])
    ax_group.set_ylabel(groupby, fontsize=10)

    # CNV heatmap
    im = ax_heat.imshow(
        mat,
        aspect="auto",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=vmin,
        vmax=vmax,
    )

    ax_heat.set_xticks([])
    ax_heat.set_yticks([])
    ax_heat.set_xlabel("Genomic position", fontsize=11)
    ax_heat.set_ylabel("Cells", fontsize=11)

    for b in boundaries:
        ax_heat.axvline(b, color="black", linewidth=0.25, alpha=0.45)

    group_vals = obs_sel[groupby].to_numpy()
    group_boundaries = np.where(group_vals[:-1] != group_vals[1:])[0] + 1

    for b in group_boundaries:
        ax_heat.axhline(b, color="black", linewidth=0.25, alpha=0.35)
        ax_group.axhline(b, color="white", linewidth=0.8)

    cb = fig.colorbar(im, cax=ax_cbar)
    cb.set_label("CNV score", fontsize=10)
    cb.ax.tick_params(labelsize=8)

    handles = [
        mpl.patches.Patch(color=color, label=label)
        for label, color in group_color_map.items()
    ]

    ax_heat.legend(
        handles=handles,
        loc="upper left",
        bbox_to_anchor=(1.04, 1.0),
        frameon=False,
        fontsize=8,
        title=groupby,
        title_fontsize=9,
    )

    fig.suptitle("InferCNV landscape", fontsize=14, fontweight="bold", y=0.985)

    pdf_path = os.path.join(outdir, f"{filename}.pdf")
    png_path = os.path.join(outdir, f"{filename}.png")

    plt.savefig(pdf_path, bbox_inches="tight")
    plt.savefig(png_path, dpi=500, bbox_inches="tight")
    plt.show()

    print(f"Saved: {pdf_path}")
    print(f"Saved: {png_path}")

In [ ]:
plot_infercnv_heatmap(
    rna,
    cnv_key="X_cnv",
    groupby="celltype",
    outdir="./fig/infercnv",
    filename="infercnv_celltype_custom",
    max_cells_per_group=800,
    vmin=-0.25,
    vmax=0.25,
    figsize=(17, 9),
)

In [ ]:
plot_infercnv_heatmap(
    rna,
    cnv_key="X_cnv",
    groupby="celltype",
    outdir="./fig/infercnv",
    filename="infercnv_celltype_custom_v015",
    max_cells_per_group=800,
    vmin=-0.15,
    vmax=0.15,
    figsize=(17, 9),
)

In [ ]:
print(rna.obsm.keys())
print(rna.layers.keys())
print(rna.uns.keys())
print(rna.var.columns.tolist())
print(rna.var.head())

In [ ]:
np.max(rna.X)

In [ ]:
np.unique(rna.obs["celltype"])

In [ ]:
cnv.tl.pca(rna)
cnv.pp.neighbors(rna)
cnv.tl.leiden(rna)

In [ ]:
cnv.pl.chromosome_heatmap(rna, groupby="cnv_leiden", dendrogram=True)

In [ ]:
cnv.tl.umap(rna)
cnv.tl.cnv_score(rna)

In [ ]:
sc.pl.umap(rna, color="cnv_score", show=False)

In [ ]:
marker_genes_dict = {
    "SFT_Tumor": ["IGF2", "STAT6"],
    "SMCs": ["RGS5", "NOTCH3", "PDGFRB", "MCAM"],
    "T": ["CD3D","CCL5", "TRBC2", "CD52"],
    "Macro":["CD68","CSF1R","TYROBP",'AIF1'],
    "SPP1+ TAMs": ["SPP1", "CCL3", "CCL4", "CX3CR1", "CSF1R"],
    "MMP9+ TAMs": ["MMP9", "ACP5", "CTSK"],
    "Endo": ["VWF", "PECAM1", "FLT1", "PLVAP"],
    "Mono": ["LYZ", "S100A4", "FCER1G", "TYROBP"],
    "Mast": ["TPSAB1", "TPSB2", "CPA3"],
    "CAFs": ["LUM", "DCN", "COL1A1", "COL3A1", "PRRX1", "SFRP2"],
    "Oligo": ["PLP1", "MAG", "MOBP"],
    "OPCs": ["PTPRZ1", "SOX10", "BCAN", "OLIG1"]
}

sc.pl.dotplot(
    rna,
    marker_genes_dict,
    groupby="celltype",
    dendrogram=False,
    cmap="YlGnBu", # https://matplotlib.org/stable/users/explain/colors/colormaps.html
    #dot_max=0.6,     # 点最大直径比例（默认 1.0）
    #dot_min=0.1,     # 点最小直径比例（默认 0）
    vmax=5,          # 显示的最大表达值，>3 的都显示为同一深红
    vmin=0,           # 最小值（默认0）
    #save="_markers.pdf"


)

In [ ]:
sc.pl.umap(rna, color=["NAB2",'STAT6'])